In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Narrow down columns to reduce GPU data movement
li = lineitem[['L_PARTKEY', 'L_SUPPKEY', 'L_ORDERKEY',
               'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_QUANTITY']]
fpart = part[['P_PARTKEY']][part.P_NAME.str.contains('ghost')]
sp = supplier[['S_SUPPKEY', 'S_NATIONKEY']]
na = nation[['N_NATIONKEY', 'N_NAME']]
ps = partsupp[['PS_PARTKEY', 'PS_SUPPKEY', 'PS_SUPPLYCOST']]
ord = orders[['O_ORDERKEY', 'O_ORDERDATE']]

# Chain merges, compute TMP and O_YEAR in one assign
df = (li.merge(fpart, left_on='L_PARTKEY', right_on='P_PARTKEY')
        .merge(sp, left_on='L_SUPPKEY', right_on='S_SUPPKEY')
        .merge(na, left_on='S_NATIONKEY', right_on='N_NATIONKEY')
        .merge(ps, left_on=['L_PARTKEY','L_SUPPKEY'], right_on=['PS_PARTKEY','PS_SUPPKEY'])
        .merge(ord, left_on='L_ORDERKEY', right_on='O_ORDERKEY')
        .assign(
            TMP=lambda x: x.L_EXTENDEDPRICE * (1 - x.L_DISCOUNT) - (x.PS_SUPPLYCOST * x.L_QUANTITY),
            O_YEAR=lambda x: x.O_ORDERDATE.dt.year
        ))

# Group and sort
total_opt = (df.groupby(['N_NAME','O_YEAR'], as_index=False, sort=False)['TMP']
           .sum()
           .sort_values(['N_NAME','O_YEAR'], ascending=[True, False]))